In [1]:
# Text Completions

import openai

# Print openai version
print(openai.__version__)

3.8.0


In [2]:
# Load the API key from the .env file
from dotenv import load_dotenv
load_dotenv()

import os
# Get the API key from the environment variables
openai.api_key = os.getenv("OPENAI_API_KEY") 
print("API Key Loaded Successfully") 

API Key Loaded Successfully


In [3]:
from openai import OpenAI
client = OpenAI(api_key=openai.api_key)

completion = client.chat.completions.create(
    model="gpt-5.5",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Write a poem about the beauty of nature."}
    ]
)

# print(completion)
print(completion.choices[0].message.content)

# There are 3 types of messages:
# 1. user - I am the user or whosoever is asking a question to the GPT model is called as the user.
# 2. assistant - GPT model is the assistant.
# 3. system - this message is the aura or the persona, which you want the GPT model to adopt while answering the question. It is optional but it can help in getting better answers from the model.

Beneath the dawn’s unfolding gold,  
The waking hills begin to glow;  
A river threads through fields of green,  
And sings to every stone below.  

The cedar lifts its quiet arms,  
The wildflower turns its painted face;  
A sparrow stitches through the air  
A hymn of sunlight, wing, and grace.  

The mountains wear the passing clouds  
Like silver shawls against the blue;  
The ocean breathes in endless waves,  
Forever old, forever new.  

In every leaf, a world is held;  
In every breeze, a soft refrain—  
Nature speaks in tender tones  
Of joy, of wonder, and of rain.  

So let us walk with gentle hearts  
Where earth and sky in beauty meet,  
And find, in all her living light,  
A sacred song beneath our feet.


## Role Prompting

Define a specific persona and expertise for the model. This activates relevant knowledge and adjusts the communication style to match the role.

**When to use:**
- You need domain-specific language and framing
- The audience for the output is specific (executives vs engineers vs customers)
- Tone and perspective matter

**Walmart use case:** Get different analyses of the same problem from different roles.


In [4]:
# Role Prompting — Same Question, Different Roles

business_problem = """
Walmart is considering deploying an AI agent to automate supplier negotiation emails. 
The agent would draft, send, and respond to routine supplier communications autonomously.
What are the key considerations?
"""

roles = {
    "AI Architect": "You are a Senior AI Systems Architect at Walmart Global Tech with 15 years of experience in enterprise AI deployments. Focus on technical architecture, reliability, and system design.",
    "Risk Officer": "You are the Chief Risk Officer at a Fortune 500 retailer. Focus on legal liability, compliance, audit trails, and reputational risk.",
    "Finance Lead": "You are VP of Finance at Walmart. Focus on ROI, cost savings, implementation costs, and financial governance.",
}

for role_name, system_prompt in roles.items():
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": business_problem},
        ],
        temperature=0.2,
        max_tokens=200,
    )
    print(f"\n{'=' * 65}")
    print(f"Role: {role_name}")
    print('=' * 65)
    print(response.choices[0].message.content.strip())



Role: AI Architect
Deploying an AI agent to automate supplier negotiation emails involves several critical considerations across various dimensions, including technical architecture, reliability, compliance, and user experience. Here are the key considerations:

### 1. **Technical Architecture**
   - **Natural Language Processing (NLP) Capabilities**: Ensure the AI agent has robust NLP capabilities to understand context, tone, and intent in supplier communications. This includes sentiment analysis and the ability to generate human-like responses.
   - **Integration with Existing Systems**: The AI agent should seamlessly integrate with existing email systems (e.g., Outlook, Gmail) and ERP systems (e.g., SAP, Oracle) to access supplier data, historical communications, and negotiation parameters.
   - **Data Storage and Management**: Implement a secure and scalable data storage solution for managing supplier information, communication history, and negotiation terms. Consider using a rela

## Structured Output Prompting

Force the model to return machine-readable JSON that conforms to a specific schema. In production, this eliminates fragile string parsing.

**The modern approach:** Use OpenAI's native structured output with Pydantic models — guaranteed schema compliance.

**Walmart use case:** Extract a shipment incident report from a free-text description.


In [6]:
from pydantic import BaseModel, Field
from typing import Optional, List
from enum import Enum

# Define the exact schema we want
class SeverityLevel(str, Enum):
    CRITICAL = "CRITICAL"
    HIGH = "HIGH"
    MEDIUM = "MEDIUM"
    LOW = "LOW"

class ShipmentIncidentReport(BaseModel):
    incident_id: str = Field(description="Generate a short unique ID like INC-001")
    sku: Optional[str] = Field(description="Product SKU if mentioned, else null")
    incident_type: str = Field(description="Category: damaged_goods, delayed_delivery, wrong_item, missing_item, other")
    severity: SeverityLevel
    financial_impact_usd: Optional[float] = Field(description="Estimated dollar impact if mentioned, else null")
    requires_immediate_action: bool
    recommended_actions: List[str] = Field(description="List of 2-3 concrete next steps")
    summary: str = Field(description="One sentence summary of the incident")

# Parse a free-text incident description
incident_text = """
Our night shift supervisor just called in. A full pallet of SKU-8832-A (Equate Vitamins, 500 count) 
got soaked during the storm because the loading dock door was left open. About 240 units are 
completely unsaleable. At $12 retail each, that's roughly $2,880 in damaged inventory. 
The supplier needs to be notified and we probably need an emergency restock request since 
we're already low on this SKU ahead of the weekend rush.
"""

response = client.beta.chat.completions.parse(
    model="gpt-4o",
    messages=[
        {"role": "system", "content": "You are a Walmart operations AI. Extract structured incident information from descriptions."},
        {"role": "user", "content": f"Parse this incident:\n\n{incident_text}"},
    ],
    response_format=ShipmentIncidentReport,
    temperature=0,
)

report = response.choices[0].message.parsed
print("Structured Incident Report:")
print("=" * 50)
print(f"Incident ID      : {report.incident_id}")
print(f"SKU              : {report.sku}")
print(f"Type             : {report.incident_type}")
print(f"Severity         : {report.severity.value}")
print(f"Financial Impact : ${report.financial_impact_usd:,.2f}" if report.financial_impact_usd else "Financial Impact : Not specified")
print(f"Immediate Action : {'WARNING  YES' if report.requires_immediate_action else 'No'}")
print(f"Summary          : {report.summary}")
print(f"\nRecommended Actions:")
for i, action in enumerate(report.recommended_actions, 1):
    print(f"  {i}. {action}")
print()
print("OK Structured output: guaranteed schema, no parsing errors.")


Structured Incident Report:
Incident ID      : INC-001
SKU              : SKU-8832-A
Type             : damaged_goods
Severity         : HIGH
Financial Impact : $2,880.00
Immediate Action : WARNING  YES
Summary          : A pallet of Equate Vitamins (SKU-8832-A) was damaged by rain, resulting in $2,880 of unsaleable inventory.

Recommended Actions:
  1. Notify the supplier about the damaged goods.
  2. Initiate an emergency restock request for SKU-8832-A.
  3. Ensure the loading dock door is secured to prevent future incidents.

OK Structured output: guaranteed schema, no parsing errors.


# Happy Learning